In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path('.').resolve().parent))

In [2]:
from src.utils.initialization import load_config

In [3]:
from pathlib import Path

cfg = load_config(Path("../configs/levi1-train.yaml"), '')
cfg.data.dataset_dir="../data/The Hague"

In [4]:
from src.models.diffusion import CityJSONDiffusionModule

ckpt = "../outputs/initial-runs/levi-1/checkpoints/last.ckpt"
model = CityJSONDiffusionModule.load_from_checkpoint(ckpt)
model.to("cuda:0")

CityJSONDiffusionModule(
  (noise): GraphNoiseModel()
  (network): rEGNNTransformer(
    (mlp_in_y): Sequential(
      (0): Linear(in_features=1, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=32, bias=True)
      (3): ReLU()
    )
    (mlp_in_X): Sequential(
      (0): Linear(in_features=5, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): ReLU()
    )
    (mlp_in_E): Sequential(
      (0): Linear(in_features=3, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=32, bias=True)
      (3): ReLU()
    )
    (mlp_in_pos): PositionsMLP(
      (mlp): Sequential(
        (0): Linear(in_features=1, out_features=16, bias=True)
        (1): ReLU()
        (2): Linear(in_features=16, out_features=1, bias=True)
      )
    )
    (tf_layers): ModuleList(
      (0-3): 4 x XEyTransformerLayer(
        (self_attn): NodeEdgeBlock(
          (in_E)

In [5]:
def _load_datamodule(cfg):
    """Datamodule for the reference-distribution metrics, or None if unavailable.

    Wasserstein/MMD/novelty need the train+test splits; validity, rejection rate and
    face coherence do not. A missing dataset degrades the run instead of killing it.
    """
    from src.utils.setup_utils import create_datamodule
    
    datamodule = create_datamodule(cfg)
    datamodule.setup()
    return datamodule

In [6]:
datamodule = _load_datamodule(cfg)

In [7]:
from src.eval.sampling import draw_samples

records, stats = draw_samples(model, 1, 10)

Face node 1 has only 0 member vertices, skipping.


In [10]:
records[1]

{'coords': array([[ 1.65633512e+00, -5.33096552e-01,  6.00192917e+00],
        [-1.31151333e-01,  2.45096207e+00,  4.69612277e+00],
        [ 1.43849456e+00, -2.77166933e-01,  5.21463431e+00],
        [-2.35084295e+00, -6.11148834e-01,  5.81188286e+00],
        [-5.29733626e-03,  5.21221757e-01,  3.86665756e+00],
        [-1.82090747e+00, -4.87043828e-01,  4.75510896e+00],
        [-1.80813134e+00, -3.01534951e-01,  4.98721135e+00],
        [-2.24592779e-02,  6.86574221e-01,  3.95798070e+00],
        [ 1.66740549e+00, -8.04128125e-02,  2.90108002e+00],
        [-3.71120945e-02,  8.85559559e-01,  4.05052257e+00],
        [-1.80540204e-01,  4.14683533e+00,  4.92123342e+00],
        [-4.14559171e-02,  9.72754300e-01,  4.08765043e+00],
        [ 1.64082038e+00, -4.11930174e-01,  5.56722844e+00],
        [ 1.49559283e+00, -1.58436984e-01,  4.15290261e+00],
        [ 1.70649648e+00, -2.60260224e-01,  5.54607857e+00],
        [-2.12924667e-02,  5.11452973e-01,  3.86054350e+00],
        [-9.18

In [16]:
import numpy as np
import plotly.io as pio
pio.renderers.default = "iframe"

# Semantic surface colours (dataviz reference palette, light mode).
_SEM_COLORS = {"GroundSurface": "#898781", "RoofSurface": "#eb6834",
               "WallSurface": "#2a78d6"}


def visualize_record(record, title=None):
    """Plotly figure of one `draw_samples` record: the reconstructed surfaces.

    Faces are shaded and outlined by semantic class; vertices are markers
    carrying their graph node id, so a malformed generated ring is visible.
    Returns a `go.Figure` -- call `.show()` in a notebook or `.write_html(path)`.
    """
    import plotly.graph_objects as go  # heavy import; only needed for plots

    cj = record["cityjson"]
    verts = np.asarray(cj["vertices"], dtype=float)
    obj_id, obj = next(iter(cj["CityObjects"].items()))
    geom = obj["geometry"][0]
    surfaces = geom["semantics"]["surfaces"]

    by_type = {}
    for face, sem in zip(geom["boundaries"][0], geom["semantics"]["values"][0]):
        by_type.setdefault(surfaces[sem]["type"], []).append(face[0])

    fig = go.Figure()
    for stype, rings in by_type.items():
        color = _SEM_COLORS.get(stype, "#eda100")
        i, j, k = [], [], []
        xs, ys, zs = [], [], []
        for ring in rings:
            # ponytail: fan triangulation, only correct for star-shaped rings --
            # switch to earcut on the fitted plane if concave faces show up.
            for t in range(1, len(ring) - 1):
                i.append(ring[0])
                j.append(ring[t])
                k.append(ring[t + 1])
            loop = verts[ring + ring[:1]]
            xs += [*loop[:, 0], None]
            ys += [*loop[:, 1], None]
            zs += [*loop[:, 2], None]
        fig.add_trace(go.Mesh3d(
            x=verts[:, 0], y=verts[:, 1], z=verts[:, 2], i=i, j=j, k=k,
            color=color, opacity=0.5, flatshading=True,
            name=f"{stype} ({len(rings)})", showlegend=True, hoverinfo="name",
        ))
        fig.add_trace(go.Scatter3d(
            x=xs, y=ys, z=zs, mode="lines", line=dict(color=color, width=3),
            showlegend=False, hoverinfo="skip",
        ))

    fig.add_trace(go.Scatter3d(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2], mode="markers",
        marker=dict(size=3, color="#0b0b0b"), name="vertices",
        hovertext=[f"v{n}<br>x={p[0]:.2f} y={p[1]:.2f} z={p[2]:.2f}"
                   for n, p in enumerate(verts)],
        hoverinfo="text", showlegend=False,
    ))

    n_faces = sum(len(r) for r in by_type.values())
    fig.update_layout(
        title=title or f"{obj_id} — {len(verts)} vertices, {n_faces} faces",
        paper_bgcolor="#fcfcfb",
        font=dict(family='system-ui, "Segoe UI", sans-serif', color="#0b0b0b"),
        legend=dict(itemsizing="constant"),
        scene=dict(aspectmode="data"),
        margin=dict(l=0, r=0, t=50, b=0),
    )
    return fig

In [21]:
fig = visualize_record(records[4])
fig.show()

In [23]:
pos, node_labels, edge_labels = model.sample(batch_size=10)

In [24]:
def visualize_sample(pos, node_labels, edge_labels, index=0, model=None):
    """Levi-graph view of one instance of a raw `model.sample(...)` batch.

    OFF nodes are dropped; face nodes are drawn at their members' centroid
    (`levi_figure`'s convention), not at the coords the model generated for them.
    Pass `model` to denormalize back to metres. Returns a `go.Figure`.
    """
    import torch

    from src.dataset.dataset import EDGE_OFF, OFF
    from src.visualize_levi import levi_figure

    p = pos[index]
    if model is not None:
        p = torch.as_tensor(model._denormalize_coords(p))
    p, nl = p.detach().cpu(), node_labels[index].detach().cpu()
    el = edge_labels[index].detach().cpu()

    keep = (nl != OFF).nonzero().flatten()
    p, nl, el = p[keep], nl[keep], el[keep][:, keep]
    ei = (el != EDGE_OFF).nonzero().t()          # [2, E], both directions
    return levi_figure({"id": f"sample_{index}", "x": p, "node_labels": nl,
                        "edge_index": ei, "edge_attr": el[ei[0], ei[1]]})

In [29]:
fig2 = visualize_sample(pos, node_labels, edge_labels, 4, model)
fig2.show()